In [1]:
!pip install opensearch-py

In [10]:
import boto3, json, time
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain.vectorstores import OpenSearchVectorSearch
from langchain.chains import RetrievalQA
from langchain_aws import ChatBedrock, ChatBedrockConverse
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.chains import RetrievalQA
from langchain_aws import BedrockEmbeddings

In [3]:
def get_data_from_s3(bucket_name, key):
    s3 = boto3.client(
        's3',
        region_name=region,
    )
    response = s3.get_object(Bucket=bucket_name, Key=key)
    data = response['Body'].read().decode('utf-8')

    return data

In [4]:
session = boto3.session.Session()
region = session.region_name
credentials = session.get_credentials()
awsauth = AWSV4SignerAuth(credentials, region, service='aoss')
aoss_client = session.client('opensearchserverless')

suffix = "demo"
bucket_name = "bucket-test-cj"
vector_store_name = f"bedrock-sample-rag-{suffix}"
index_name = f"bedrock-sample-index-{suffix}"

In [5]:
# Retrieve collection details programmatically
collection_response = aoss_client.batch_get_collection(names=[vector_store_name])
if 'collectionDetails' in collection_response and collection_response['collectionDetails']:
    collection_id = collection_response['collectionDetails'][0]['id']
    host = f"{collection_id}.{region}.aoss.amazonaws.com"
    print("Using host:", host)
else:
    raise ValueError("Could not retrieve collection details.")

Using host: 3e7z7ay9fy9a8vlrk4j4.us-west-2.aoss.amazonaws.com


In [6]:
oss_client = OpenSearch(
    hosts=[{'host': host, 'port': 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)

In [7]:
s3_data = get_data_from_s3("bucket-test-cj", "full_text.txt")
s3_data[0:200]

'BILL Reports Second Quarter Fiscal Year 2025 Financial Results\n• Q2 Core Revenue Increased 16% Year-Over-Year\n• Q2 Total Revenue Increased 14% Year-Over-Year\nSAN JOSE, Calif.--(BUSINESS WIRE) – Februa'

In [17]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.create_documents([s3_data])
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 40


In [11]:
embeddings = BedrockEmbeddings(model_id="amazon.titan-embed-text-v2:0")

In [18]:
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)

In [26]:
docs = vector_store.similarity_search("What were the subscription fees?", k=3)

for i, doc in enumerate(docs, start=1):
    print(f"=== Document {i} ===")
    print("Content:")
    print(doc.page_content)

=== Document 1 ===
Content:
to-day financial workflow. We are moving fast to address a vast market opportunity to transform the financial operations for millions 
of SMBs.”
“In Q2, we delivered strong financial results, expanded our non-GAAP operating margin, and continued our track record of 
execution across the company,” said John Rettig, BILL President and CFO. “We are executing on our strategic priorities and are 
confident that our strong business model will allow us to drive years of durable growth, an attractive long-term profitability profile, 
and sustained value generation for shareholders.”
Financial Highlights for the Second Quarter of Fiscal 2025:
• Total revenue was $362.6 million, an increase of 14% year-over-year.
• Core revenue, which consists of subscription and transaction fees, was $319.6 million, an increase of 16% year-over-
year. Subscription fees were $67.7 million, up 7% year-over-year. Transaction fees were $251.9 million, up 19% year-
over-year.
=== Document

In [28]:
llm = ChatBedrockConverse(
    model="anthropic.claude-3-sonnet-20240229-v1:0",
    temperature=0.0,
    top_p=0.9,
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_kwargs={"k": 3})
)

In [29]:
qa_chain.run('What were the subscription fees?')

/tmp/ipykernel_27221/3057035564.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  qa_chain.run('What were the subscription fees?')


'According to the financial highlights provided, the subscription fees for the second quarter of fiscal 2025 were $67.7 million, up 7% year-over-year.'